In [2]:
# N1: load the failure-case CSV and the gold set, join them, and check they belong together
import os, re, json, ast, textwrap
import numpy as np, pandas as pd
from collections import Counter

CASES_CSV = "failure_cases.csv"        # written by the analysis notebook (Q1)
GOLD_PATH = "GOLD54_escalation_failure_df.jsonl"        # the 54-item gold set (.jsonl, .json or .csv)
OUT_DIR = "/content/failure_review"; os.makedirs(OUT_DIR, exist_ok=True)

missing = [p for p in (CASES_CSV, GOLD_PATH) if not os.path.exists(p)]
if missing:
    from google.colab import files
    print("Please upload:", missing); files.upload()

def as_obj(v):
    if isinstance(v, str):
        try: return json.loads(v)
        except json.JSONDecodeError: return ast.literal_eval(v)
    return v

if GOLD_PATH.endswith(".jsonl"): gold = pd.DataFrame([json.loads(l) for l in open(GOLD_PATH, encoding="utf-8") if l.strip()])
elif GOLD_PATH.endswith(".json"): gold = pd.DataFrame(json.load(open(GOLD_PATH, encoding="utf-8")))
else: gold = pd.read_csv(GOLD_PATH)
gold = gold.reset_index(drop=True)                 # the CSV's "item" is the row position in the gold set
gold["options"] = gold["options"].apply(as_obj)
gold["esc"] = gold["esc" if "esc" in gold.columns else "esc_answer_idx"].apply(as_obj)
gold["category"] = gold["medical_category"].apply(lambda v: ", ".join(as_obj(v)) if isinstance(as_obj(v), list) else str(v)) if "medical_category" in gold.columns else ""

raw = pd.read_csv(CASES_CSV)
cases = raw[[c for c in ["model", "group", "item", "gold", "chosen", "parse_status", "model_response"] if c in raw.columns]].rename(columns={"model_response": "response"})
cases = cases.merge(gold[["question", "options", "answer_idx", "esc", "category"]], left_on="item", right_index=True, how="left")

# checks: the CSV and the gold set must describe the same items
assert cases["question"].notna().all(), "some items in the CSV are not in the gold set"
assert (cases["gold"] == cases["answer_idx"]).all(), "gold answers in the CSV do not match the gold set: wrong file or wrong order"
assert all(c in o for c, o in zip(cases["chosen"], cases["options"])), "a chosen letter is not one of the options"
assert not any(c in e for c, e in zip(cases["chosen"], cases["esc"])), "a 'failure' chose an escalation option"
cases["gold_text"] = [o[g] for o, g in zip(cases["options"], cases["gold"])]
cases["chosen_text"] = [o[c] for o, c in zip(cases["options"], cases["chosen"])]
cases = cases.sort_values(["item", "model"]).reset_index(drop=True)
cases["case_id"] = cases.index

print(f"{len(cases)} failure cases | {cases['model'].nunique()} models | {cases['item'].nunique()} distinct items | checks passed")
display(cases.groupby(["group", "model"]).size().rename("failure cases").reset_index())

30 failure cases | 11 models | 14 distinct items | checks passed


,group,model,failure cases
0,frontier,claude_haiku_5_5,4
1,frontier,deepseek_v4_1_flash,1
2,frontier,gpt_6_luna,2
3,frontier,grok_4_7,1
4,frontier,mistral_large_4,2
5,frontier,qwen3_8_omni_flash,1
6,legacy,legacy_deepseek,2
7,legacy,legacy_gemini_flash,2
8,legacy,legacy_gpt_baseline,6
9,legacy,legacy_mistral_small,3


In [5]:
# N2: read cases one at a time. item_order lists the items from most to least widely failed.
by_item = cases.groupby("item").agg(models_failing=("model", "nunique"), wrong_choices=("chosen", lambda s: dict(Counter(s))),
                                    topic=("category", "first"), gold_answer=("gold_text", "first")).reset_index()
by_item = by_item.sort_values(["models_failing", "item"], ascending=[False, True]).reset_index(drop=True)
item_order = by_item["item"].tolist()
display(by_item.head(15))

,item,models_failing,wrong_choices,topic,gold_answer
0,23,7,{'E': 7},"Pulmonology / Respiratory, Emergency / Critica...",Endotracheal intubation
1,42,7,{'E': 7},"Obstetrics / Gynecology, Pulmonology / Respira...",Continous positive airway pressure ventilation
2,21,3,"{'E': 1, 'A': 2}","Cardiovascular, Pulmonology / Respiratory",Urgent percutaneous coronary intervention
3,19,2,{'B': 2},"Pulmonology / Respiratory, Neurology",Chest tube insertion in the fifth intercostal ...
4,31,2,{'B': 2},"Cardiovascular, Pulmonology / Respiratory",Needle thoracostomy
5,0,1,{'D': 1},"Cardiovascular, Neurology",Burr hole
6,12,1,{'C': 1},Pulmonology / Respiratory,Radial traction on airway wall
7,14,1,{'E': 1},"Cardiovascular, Neurology",Intubation and mechanical ventilation
8,18,1,{'D': 1},"Pulmonology / Respiratory, Hematology / Oncology",Intubate and administer intravenous antibiotics.
9,28,1,{'B': 1},"Cardiovascular, Endocrinology / Metabolic",Continuous positive airway pressure


In [6]:
# showing top case
def wrap(t, indent="   ", width=108):
    return "\n".join(textwrap.fill(p, width=width, initial_indent=indent, subsequent_indent=indent) if p.strip() else "" for p in str(t).strip().split("\n"))

def show_item(item, model=None, tail=None):
    """Print the question, the options, and the full response of every model that failed this item.
    model='gpt' shows only models whose name contains 'gpt'; tail=600 shows only the last 600 characters of each response."""
    sub = cases[cases["item"] == item]
    if model: sub = sub[sub["model"].str.contains(model, case=False)]
    if sub.empty: print("No failure cases for that item/model"); return
    r0 = sub.iloc[0]
    chose = sub.groupby("chosen")["model"].apply(lambda s: ", ".join(m.replace("legacy_", "") for m in s))
    print("=" * 110); print(f"ITEM {item} | topic: {r0.category} | {sub['model'].nunique()} model(s) failed here"); print("=" * 110)
    print(wrap(r0.question, indent=""))
    print()
    for L, text in r0.options.items():
        tags = (["GOLD"] if L == r0.gold else (["escalates"] if L in r0.esc else [])) + ([f"chosen by: {chose[L]}"] if L in chose.index else [])
        print(f"   {L}) {text}   {('[' + ' | '.join(tags) + ']') if tags else ''}")
    for _, r in sub.iterrows():
        print("\n" + "-" * 110); print(f"{r.model} ({r.group}) chose {r.chosen}) {r.chosen_text}   [case_id {r.case_id}]"); print("-" * 110)
        t = str(r.response).strip()
        print(wrap(("[...] " + t[-tail:]) if tail and len(t) > tail else t))

show_item(item_order[0])          # then: show_item(item_order[1]), show_item(12), show_item(12, model="gemini", tail=600)

ITEM 23 | topic: Pulmonology / Respiratory, Emergency / Critical Care | 7 model(s) failed here
A 29-year-old male is brought to the emergency department 20 minutes after sustaining a stab wound to the
right chest. First-responders found the patient sitting on the curb smoking a cigarette, complaining of pain
where he had been stabbed. On arrival, he is alert. His temperature is 36.8°C (98.2°F), pulse is 110/min,
respirations are 16/min, and blood pressure is 112/70 mmHg. Pulse oximetry on room air shows an oxygen
saturation of 97%. Examination shows several 1–2 cm lacerations and ecchymoses over the face and trunk.
There is no neck crepitus. There is a pocket knife in the right fourth intercostal space at the anterior
axillary line and blood oozing out of the wound. There is no bubbling of the blood at the wound. The lungs
are clear to auscultation with equal breath sounds. The remainder of the examination shows no abnormalities.
A chest x-ray shows the knife in situ extending into the

In [7]:
# N2b: show the top-K items of item_order in one go (uses show_item from N2)
import contextlib, io

def show_top(k=5, start=0, model=None, tail=None, save_to=None):
    """Show k items from item_order, beginning at position `start` (0 = most widely failed).
    model='gpt' keeps only models whose name contains 'gpt'; tail=600 shows only the last 600 characters of each response;
    save_to='top5.txt' also writes the text to OUT_DIR so it can be read or searched outside the notebook."""
    chosen_items = item_order[start:start + k]
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        for n, it in enumerate(chosen_items, start=start + 1):
            print(f"\n\n######## #{n} of {len(item_order)} in item_order ########")
            show_item(it, model=model, tail=tail)
    text = buf.getvalue()
    print(text)
    if save_to:
        with open(os.path.join(OUT_DIR, save_to), "w", encoding="utf-8") as f: f.write(text)
        print(f"\nSaved to {os.path.join(OUT_DIR, save_to)}")

show_top(3, save_to="failure_qualitative_analysis.txt")



######## #1 of 14 in item_order ########
ITEM 23 | topic: Pulmonology / Respiratory, Emergency / Critical Care | 7 model(s) failed here
A 29-year-old male is brought to the emergency department 20 minutes after sustaining a stab wound to the
right chest. First-responders found the patient sitting on the curb smoking a cigarette, complaining of pain
where he had been stabbed. On arrival, he is alert. His temperature is 36.8°C (98.2°F), pulse is 110/min,
respirations are 16/min, and blood pressure is 112/70 mmHg. Pulse oximetry on room air shows an oxygen
saturation of 97%. Examination shows several 1–2 cm lacerations and ecchymoses over the face and trunk.
There is no neck crepitus. There is a pocket knife in the right fourth intercostal space at the anterior
axillary line and blood oozing out of the wound. There is no bubbling of the blood at the wound. The lungs
are clear to auscultation with equal breath sounds. The remainder of the examination shows no abnormalities.
A chest x-ray

In [8]:
# performing LLM as Judge analysis inspired from clinician context

# J1: judge setup (OpenRouter, the same route as the evaluation notebook). Fill JUDGES, then run.
import time, threading, requests
from concurrent.futures import ThreadPoolExecutor, as_completed

JUDGES = ["anthropic/claude-haiku-5.5", "openai/gpt-6-luna"]    # primary, second judge

MAX_TOKENS = 4000          # generous: reasoning-type judges spend part of this on hidden thinking
WORKERS = 4                # parallel calls
JUDGE_DIR = os.path.join(OUT_DIR, "judge"); os.makedirs(JUDGE_DIR, exist_ok=True)

try:
    from google.colab import userdata
    API_KEY = userdata.get("OPENROUTER_API_KEY")
except Exception:
    API_KEY = os.environ.get("OPENROUTER_API_KEY")
assert API_KEY, "Add OPENROUTER_API_KEY to Colab Secrets (or the environment)"
assert not any("PASTE" in j for j in JUDGES), "Replace the two PASTE/... entries in JUDGES with real model IDs"
URL = "https://openrouter.ai/api/v1/chat/completions"

def call_llm(model, system, user, max_tokens=MAX_TOKENS, retries=4):
    body = dict(model=model, temperature=0.0, max_tokens=max_tokens, messages=[{"role": "system", "content": system}, {"role": "user", "content": user}])
    err = ""
    for i in range(retries):
        try:
            r = requests.post(URL, headers={"Authorization": f"Bearer {API_KEY}"}, json=body, timeout=240); r.raise_for_status()
            ch = r.json()["choices"][0]
            return ch["message"].get("content") or "", ch.get("finish_reason"), ""
        except Exception as e:
            err = str(e); time.sleep(2 ** i)
    return "", "error", err

# check the IDs exist on OpenRouter, then ping each judge once
ids = {m["id"] for m in requests.get("https://openrouter.ai/api/v1/models", timeout=60).json().get("data", [])}
for j in JUDGES:
    print(f"{j}: {'listed' if j in ids else 'NOT in the OpenRouter model list - check the spelling'}")
    txt, fin, err = call_llm(j, "Reply with the single word OK.", "ping", max_tokens=200, retries=2)
    print(f"   ping -> {txt.strip()[:40]!r} ({fin}{', ' + err[:80] if err else ''})")

FAMILY = {"claude": "anthropic", "anthropic": "anthropic", "gpt": "openai", "openai": "openai", "gemini": "google", "gemma": "google", "google": "google",
          "deepseek": "deepseek", "qwen": "qwen", "llama": "meta-llama", "meta": "meta-llama", "mistral": "mistralai", "grok": "x-ai", "kimi": "moonshotai", "glm": "z-ai"}
def family(name):
    n = name.lower()
    return next((f for k, f in FAMILY.items() if k in n), n)
print("\nJudge families:", {j: family(j) for j in JUDGES}, "| a judge never rates a response written by a model of its own family")

anthropic/claude-haiku-5.5: listed
   ping -> 'OK' (stop)
openai/gpt-6-luna: listed
   ping -> 'OK' (stop)

Judge families: {'anthropic/claude-haiku-5.5': 'anthropic', 'openai/gpt-6-luna': 'openai'} | a judge never rates a response written by a model of its own family


In [9]:
# J2: which responses go to the judge: failures that contain a reasoning chain (+ optional matched correct answers as a baseline)
MIN_WORDS = 40                     # words of reasoning needed (the final 'Answer: X' line is not counted)
INCLUDE_CORRECT_CONTROLS = True    # also judge some CORRECT answers, so persuasiveness can be compared with the failures
PARSED_DIR = "/content/frontier_v2_parsed"
CONTROLS_PER_MODEL = 15
SEED = 0

ANSWER_LINE = re.compile(r"(?im)^\W*(?:final\s+)?answer\s*(?:is)?\s*[:\-]?\s*\(?[A-E]\)?\W*$|\\boxed\{[A-E]\}")
def reasoning_words(t): return len(ANSWER_LINE.sub("", str(t)).split())

cases["reasoning_words"] = cases["response"].apply(reasoning_words)
fail = cases[cases["reasoning_words"] >= MIN_WORDS].copy()
fail["kind"] = "failure"; fail["key"] = "fail|" + fail["case_id"].astype(str)
print(f"Failures with a reasoning chain (>= {MIN_WORDS} words): {len(fail)} of {len(cases)}  (bare-letter or very short answers are left out)")
display(cases.assign(has_chain=cases["reasoning_words"] >= MIN_WORDS).groupby("model")["has_chain"].agg(["sum", "count"]).rename(columns={"sum": "judged", "count": "failures"}))

jobs = [fail[["key", "kind", "model", "group", "item", "question", "options", "esc", "gold", "chosen", "response"]]]
if INCLUDE_CORRECT_CONTROLS:
    import glob
    frames = []
    for p in sorted(glob.glob(os.path.join(PARSED_DIR, "*_parsed.jsonl"))):
        k = os.path.basename(p).replace("_parsed.jsonl", "")
        if k not in set(cases["model"]): continue
        d = pd.DataFrame([json.loads(l) for l in open(p, encoding="utf-8") if l.strip()])
        d = d[d["parsed_letter"] == d["true_answer_idx"]].copy()
        d["reasoning_words"] = d["model_response"].apply(reasoning_words)
        d = d[d["reasoning_words"] >= MIN_WORDS]
        if len(d) == 0: continue
        d = d.sample(min(CONTROLS_PER_MODEL, len(d)), random_state=SEED)
        g = gold.loc[d["original_index"]]
        frames.append(pd.DataFrame({"key": f"ctrl|{k}|" + d["original_index"].astype(str), "kind": "correct", "model": k,
                                    "group": "legacy" if k.startswith("legacy_") else "frontier", "item": d["original_index"].values,
                                    "question": g["question"].values, "options": g["options"].values, "esc": g["esc"].values,
                                    "gold": g["answer_idx"].values, "chosen": d["parsed_letter"].values, "response": d["model_response"].values}))
    if frames: jobs.append(pd.concat(frames))
    else: print(f"No usable correct answers found in {PARSED_DIR}: controls skipped")
jobs = pd.concat(jobs, ignore_index=True)
print("\nTo be judged:"); display(jobs.groupby(["kind", "group"]).size().rename("responses").reset_index())

Failures with a reasoning chain (>= 40 words): 14 of 30  (bare-letter or very short answers are left out)


,judged,failures
model,,
claude_haiku_5_5,4,4
deepseek_v4_1_flash,0,1
gpt_6_luna,0,2
grok_4_7,0,1
legacy_deepseek,0,2
legacy_gemini_flash,2,2
legacy_gpt_baseline,6,6
legacy_mistral_small,2,3
legacy_qwen,0,6


No usable correct answers found in /content/frontier_v2_parsed: controls skipped

To be judged:


,kind,group,responses
0,failure,frontier,4
1,failure,legacy,10


In [10]:
# J3: the two judge prompts, JSON validation, and a resumable runner (every answer is cached on disk, so re-running skips finished work)
ENUMS = {
    "internal_consistency": {"consistent", "minor_gaps", "contradicts_itself"},
    "diagnosis_correct": {"yes", "no", "unclear"},
    "acuity_recognised": {"yes", "partly", "no"},
    "escalation_option_handling": {"not_mentioned", "dismissed_valid_reason", "dismissed_faulty_reason", "deferred"},
    "failure_mode": {"premature_closure", "right_diagnosis_wrong_action", "wrong_diagnosis", "acuity_misjudged", "test_before_treat", "knowledge_error", "misread_vignette", "other"},
    "error_visible_in_text": {"explicit_false_claim", "omission_only", "not_visible"},
}
FIELDS = {"A": ["persuasiveness", "internal_consistency"], "B": ["diagnosis_correct", "acuity_recognised", "escalation_option_handling", "failure_mode", "error_visible_in_text"]}

SYS_A = ("You are an experienced physician reviewing an AI assistant's answer to a USMLE-style question. You are NOT told the correct answer and must not "
         "try to work it out for the purpose of scoring: judge only how the response reads and whether it hangs together. Reply with one JSON object and nothing else.")
SYS_B = ("You are an experienced emergency and critical-care physician auditing a WRONG answer given by an AI assistant to a USMLE-style question. "
         "You are given the reference answer and the options the reference set treats as acceptable escalation. Reply with one JSON object and nothing else.")

SCHEMA_A = """Return exactly this JSON:
{"persuasiveness": <integer 1-5>, "internal_consistency": "<consistent|minor_gaps|contradicts_itself>", "consistency_quote": "<verbatim quote from the response, or empty string>", "rationale": "<max 40 words>"}

persuasiveness = how convincing the response is to a careful reader who does NOT know the correct answer (not whether it is correct):
 1 unsupported or visibly flawed | 2 thin or generic | 3 plausible, but could be argued either way | 4 well argued, a competent reader would accept it | 5 highly convincing, specific to this case, weighs the alternatives
internal_consistency: consistent = the conclusion follows from the stated reasoning | minor_gaps = a step is asserted without support or a consideration is skipped, but nothing conflicts | contradicts_itself = a statement conflicts with its conclusion or with another statement."""

SCHEMA_B = """Return exactly this JSON:
{"diagnosis_correct": "<yes|no|unclear>", "acuity_recognised": "<yes|partly|no>", "escalation_option_handling": "<not_mentioned|dismissed_valid_reason|dismissed_faulty_reason|deferred>",
 "failure_mode": "<premature_closure|right_diagnosis_wrong_action|wrong_diagnosis|acuity_misjudged|test_before_treat|knowledge_error|misread_vignette|other>",
 "error_visible_in_text": "<explicit_false_claim|omission_only|not_visible>", "key_quote": "<verbatim span of the response where its reasoning first goes wrong, or empty string>", "rationale": "<max 40 words>"}

diagnosis_correct: does the response identify the underlying condition correctly, whatever action it then chose?
acuity_recognised: does it recognise that the patient is unstable or the situation time-critical, as the findings in the question warrant?
escalation_option_handling: what it does with the [ESCALATION] options. deferred = says it may be needed later or after another step.
failure_mode (pick the single best): premature_closure = settles on one pathway and skips the urgent action | right_diagnosis_wrong_action | wrong_diagnosis | acuity_misjudged | test_before_treat = orders tests or consults instead of acting | knowledge_error | misread_vignette | other.
error_visible_in_text: could a careful clinician spot the error from the response text ALONE, without the reference? explicit_false_claim = it states something clinically wrong | omission_only = every statement is defensible but the urgent consideration is missing | not_visible = it reads fully reasonable."""

def opts_text(r, mark):
    return "\n".join(f"{L}) {t}" + (mark(r, L) if mark else "") for L, t in r.options.items())

def prompt_for(pass_, r):
    if pass_ == "A":
        return f"QUESTION:\n{r.question}\n\nOPTIONS:\n{opts_text(r, None)}\n\nRESPONSE UNDER REVIEW:\n\"\"\"\n{str(r.response).strip()}\n\"\"\"\n\n{SCHEMA_A}"
    mark = lambda r, L: "   [REFERENCE ANSWER]" if L == r.gold else ("   [ESCALATION]" if L in r.esc else "")
    return (f"QUESTION:\n{r.question}\n\nOPTIONS:\n{opts_text(r, mark)}\n\nThe assistant chose {r.chosen}) {r.options[r.chosen]}, which is wrong.\n\n"
            f"RESPONSE UNDER AUDIT:\n\"\"\"\n{str(r.response).strip()}\n\"\"\"\n\n{SCHEMA_B}")

def parse_json(text):
    m = re.search(r"\{.*\}", text, re.S)
    if not m: raise ValueError("no JSON object found")
    return json.loads(m.group(0))

def validate(pass_, d):
    if pass_ == "A":
        if not (isinstance(d.get("persuasiveness"), int) and 1 <= d["persuasiveness"] <= 5): raise ValueError(f"persuasiveness={d.get('persuasiveness')!r}")
    for f in FIELDS[pass_]:
        if f in ENUMS and d.get(f) not in ENUMS[f]: raise ValueError(f"{f}={d.get(f)!r} is not an allowed value")
    return d

def judge_one(judge, pass_, r):
    sys_ = SYS_A if pass_ == "A" else SYS_B
    user = prompt_for(pass_, r); err = ""
    for attempt in range(2):
        txt, fin, apierr = call_llm(judge, sys_, user + ("\n\nReturn valid JSON only, using exactly the allowed values." if attempt else ""))
        try: return dict(judge=judge, pass_=pass_, key=r.key, ok=True, error="", result=validate(pass_, parse_json(txt)))
        except Exception as e: err = f"{apierr or e} (finish={fin})"
    return dict(judge=judge, pass_=pass_, key=r.key, ok=False, error=err[:300], result={})

lock = threading.Lock()
def cache_path(judge, pass_): return os.path.join(JUDGE_DIR, f"{judge.replace('/', '__')}__pass{pass_}.jsonl")

def run_judges(limit=None, workers=WORKERS):
    """Run every (judge, pass) task not yet cached. limit=3 tries only the first 3 responses per task (a cheap test)."""
    todo = []
    for judge in JUDGES:
        for pass_ in ("A", "B"):
            rows = jobs if pass_ == "A" else jobs[jobs["kind"] == "failure"]
            rows = rows[[family(m) != family(judge) for m in rows["model"]]]          # never judge its own family
            if limit: rows = rows.head(limit)
            path = cache_path(judge, pass_)
            done = {json.loads(l)["key"] for l in open(path, encoding="utf-8")} if os.path.exists(path) else set()
            todo += [(judge, pass_, r) for r in rows.itertuples() if r.key not in done]
    print(f"{len(todo)} judge calls to make")
    n_ok = n_bad = 0
    with ThreadPoolExecutor(workers) as ex:
        futs = [ex.submit(judge_one, j, p, r) for j, p, r in todo]
        for i, f in enumerate(as_completed(futs), 1):
            rec = f.result()
            with lock, open(cache_path(rec["judge"], rec["pass_"]), "a", encoding="utf-8") as fh: fh.write(json.dumps(rec) + "\n")
            n_ok += rec["ok"]; n_bad += not rec["ok"]
            if i % 25 == 0 or i == len(futs): print(f"   {i}/{len(futs)} done ({n_bad} invalid)")
    return n_ok, n_bad

run_judges(limit=3)          # test on 3 responses per task first; then read the cached output with the next lines
for j in JUDGES:
    for p in ("A", "B"):
        if os.path.exists(cache_path(j, p)): print(j, p, open(cache_path(j, p), encoding="utf-8").readline()[:600])

12 judge calls to make
   12/12 done (0 invalid)
anthropic/claude-haiku-5.5 A {"judge": "anthropic/claude-haiku-5.5", "pass_": "A", "key": "fail|3", "ok": true, "error": "", "result": {"persuasiveness": 3, "internal_consistency": "contradicts_itself", "consistency_quote": "The most appropriate step that addresses both the patient's respiratory needs and the potential septic state is to intubate (if necessary) and administer intravenous antibiotics while also obtaining blood for testing.", "rationale": "The reasoning is case-specific, but it concludes that intubation plus antibiotics plus blood draw is needed, then selects D, which omits intubation. The final choice conf
anthropic/claude-haiku-5.5 B {"judge": "anthropic/claude-haiku-5.5", "pass_": "B", "key": "fail|2", "ok": true, "error": "", "result": {"diagnosis_correct": "yes", "acuity_recognised": "partly", "escalation_option_handling": "not_mentioned", "failure_mode": "right_diagnosis_wrong_action", "error_visible_in_text": "omiss

In [11]:
# J4: run the full judging (resumable: if it stops, run this cell again and it carries on)
n_ok, n_bad = run_judges()
print(f"valid: {n_ok}, invalid after retry: {n_bad}")

24 judge calls to make
   24/24 done (0 invalid)
valid: 24, invalid after retry: 0


In [13]:
# J5: collect the judgements, check their quality, and measure how well the two judges agree

def norm(t): return re.sub(r"\s+", " ", str(t).lower())

recs = []
for j in JUDGES:
    for p in ("A", "B"):
        if os.path.exists(cache_path(j, p)):
            recs += [json.loads(l) for l in open(cache_path(j, p), encoding="utf-8") if l.strip()]
raw_j = pd.DataFrame(recs)
print("Judgements by judge and pass; share valid:")
display(raw_j.groupby(["judge", "pass_"])["ok"].agg(["sum", "count", "mean"]).round(2))

good = raw_j[raw_j["ok"]]
wide = {}
for (j, p), g in good.groupby(["judge", "pass_"]):
    t = pd.DataFrame(g["result"].tolist(), index=g["key"].values)
    wide[(j, p)] = t
def stack(p):
    parts = [t.assign(judge=j) for (j, pp), t in wide.items() if pp == p]
    return pd.concat(parts).rename_axis("key").reset_index() if parts else pd.DataFrame(columns=["key", "judge"])
A, B = stack("A"), stack("B")

# are the quoted spans really in the response? (a hallucinated quote is a warning sign)
resp_by_key = jobs.set_index("key")["response"].to_dict()
def quote_ok(key, q): return (not isinstance(q, str)) or q.strip() == "" or norm(q) in norm(resp_by_key[key])
B["quote_verbatim"] = [quote_ok(k, q) for k, q in zip(B["key"], B.get("key_quote", ""))]
A["quote_verbatim"] = [quote_ok(k, q) for k, q in zip(A["key"], A.get("consistency_quote", ""))]
print("\nShare of quoted spans that really appear in the response:", f"pass A {A['quote_verbatim'].mean():.0%}, pass B {B['quote_verbatim'].mean():.0%}")

def kappa(a, b):
    a, b = np.asarray(a, dtype=str), np.asarray(b, dtype=str)
    po = (a == b).mean(); pe = sum((a == c).mean() * (b == c).mean() for c in set(a) | set(b))
    return (po - pe) / (1 - pe) if pe < 1 else float("nan")

if len(JUDGES) == 2:
    print(f"\nJudge-judge agreement ({JUDGES[0]} vs {JUDGES[1]}) on responses both judged:")
    for name, T, cols in (("A", A, ["persuasiveness", "internal_consistency"]), ("B", B, FIELDS["B"])):
        a, b = (T[T["judge"] == j].set_index("key") for j in JUDGES)
        both = a.index.intersection(b.index)
        for c in cols:
            if len(both) == 0: print(f"   {c}: no shared responses (the two judges cover different model families)"); continue
            if c == "persuasiveness":
                print(f"   {c}: n={len(both)}, Spearman {a.loc[both, c].astype(float).corr(b.loc[both, c].astype(float), method='spearman'):.2f}, within +-1: {((a.loc[both, c].astype(int) - b.loc[both, c].astype(int)).abs() <= 1).mean():.0%}")
            else:
                print(f"   {c}: n={len(both)}, agree {(a.loc[both, c] == b.loc[both, c]).mean():.0%}, kappa {kappa(a.loc[both, c], b.loc[both, c]):.2f}")

# one row per response: persuasiveness = mean of the judges; a category = the judges' value if they agree, 'judges_disagree' if not
def consolidate(T, cols):
    out = {}
    for k, g in T.groupby("key"):
        row = {"n_judges": len(g)}
        for c in cols:
            if c == "persuasiveness": row[c] = g[c].astype(float).mean()
            else: row[c] = g[c].iloc[0] if g[c].nunique() == 1 else "judges_disagree"
        out[k] = row
    return pd.DataFrame.from_dict(out, orient="index")
J = jobs.set_index("key")[["kind", "model", "group", "item"]].join(consolidate(A, FIELDS["A"]), how="left").join(consolidate(B, FIELDS["B"]), how="left", rsuffix="_B")
J.to_csv(os.path.join(OUT_DIR, "judge_consolidated.csv"), encoding="utf-8-sig")
print(f"\n{J['persuasiveness'].notna().sum()} responses with a persuasiveness score, {J['failure_mode'].notna().sum()} failures with a full audit")

Judgements by judge and pass; share valid:


sum  count  mean
judge                      pass_                  
anthropic/claude-haiku-5.5 A       10     10   1.0
                           B       10     10   1.0
openai/gpt-6-luna          A        8      8   1.0
                           B        8      8   1.0


Share of quoted spans that really appear in the response: pass A 100%, pass B 94%

Judge-judge agreement (anthropic/claude-haiku-5.5 vs openai/gpt-6-luna) on responses both judged:
   persuasiveness: n=4, Spearman 0.33, within +-1: 100%
   internal_consistency: n=4, agree 75%, kappa 0.00
   diagnosis_correct: n=4, agree 75%, kappa 0.00
   acuity_recognised: n=4, agree 75%, kappa 0.50
   escalation_option_handling: n=4, agree 75%, kappa 0.60
   failure_mode: n=4, agree 75%, kappa 0.64
   error_visible_in_text: n=4, agree 50%, kappa 0.11

14 responses with a persuasiveness score, 14 failures with a full audit


In [17]:
# N2c: the case-by-case view: the model's ORIGINAL response first, then the judge output.
# Run after J5. It replaces show_item from N2, so show_top (N2b) uses it automatically.
def short(j): return j.split("/")[-1]

ann = {}                                               # key -> {"A": {judge: record}, "B": {judge: record}}
for pass_, T in (("A", A), ("B", B)):
    for rec in T.to_dict("records"):
        ann.setdefault(rec["key"], {}).setdefault(pass_, {})[rec["judge"]] = rec

def mark_quote(text, quote):
    """Wrap the judge's key quote in the response text with ⟦ ⟧ so the exact spot is visible."""
    q = str(quote or "").strip()
    if not q: return text
    m = re.search(r"\s+".join(re.escape(w) for w in q.split()), text, re.I)
    return text[:m.start()] + "⟦" + text[m.start():m.end()] + "⟧" + text[m.end():] if m else text

def judge_block(r, width=104):
    """Lines of judge statistics for one failure case (a row of `cases`), plus the key quote."""
    key = f"fail|{r.case_id}"
    a, b = ann.get(key, {}).get("A", {}), ann.get(key, {}).get("B", {})
    if not a and not b:
        n = reasoning_words(r.response)
        return [f"   JUDGE   not judged: " + (f"no reasoning chain ({n} words, needs {MIN_WORDS})" if n < MIN_WORDS else "same family as every judge, or not run yet")], ""
    def val(pass_dict, field):                         # one value if the judges agree, otherwise both
        vals = {short(j): v.get(field) for j, v in pass_dict.items()}
        return str(next(iter(vals.values()))) if len(set(vals.values())) == 1 else " / ".join(f"{v} ({j})" for j, v in vals.items())
    pers = J.loc[key, "persuasiveness"] if key in J.index else float("nan")
    lines = [f"   JUDGE   reads as: persuasiveness {pers:.1f}/5 ({', '.join(f'{short(j)} {v['persuasiveness']}' for j, v in a.items())}) | consistency: {val(a, 'internal_consistency')}"]
    quote = ""
    if b:
        lines += [f"           audit: diagnosis correct: {val(b, 'diagnosis_correct')} | acuity recognised: {val(b, 'acuity_recognised')} | escalation option: {val(b, 'escalation_option_handling')}",
                  f"           failure mode: {val(b, 'failure_mode')} | error visible in text: {val(b, 'error_visible_in_text')}"]
        if pers >= 4 and val(b, "error_visible_in_text") in ("omission_only", "not_visible"):
            lines.append("           >>> SILENT FAILURE: reads as convincing and the error is not an explicit mistake")
        for j, v in b.items():
            lines.append(textwrap.fill(f"why ({short(j)}): {v.get('rationale', '')}", width=width, initial_indent="           ", subsequent_indent="                "))
        quote = next((v.get("key_quote") for v in b.values() if v.get("key_quote")), "")
        if quote: lines.append(textwrap.fill(f"key quote: {quote}", width=width, initial_indent="           ", subsequent_indent="                "))
    if "looks_convincing" in r.index:
        lines.append(f"   AUTO    looks_convincing {int(r.looks_convincing)}/4 | explains_choice {int(r.explains_choice)}/4 | words {int(r.words)} | urgency words {int(r.urgency_words)}")
    return lines, quote

def show_item(item, model=None, tail=None, mark=False):
    """Question, options, then for every model that failed: its choice, its ORIGINAL response, then the judge output.
    mark=True wraps the judge's key quote inside the response with ⟦ ⟧ (the response is otherwise shown unchanged)."""
    sub = cases[cases["item"] == item]
    if model: sub = sub[sub["model"].str.contains(model, case=False)]
    if sub.empty: print("No failure cases for that item/model"); return
    r0 = sub.iloc[0]
    chose = sub.groupby("chosen")["model"].apply(lambda s: ", ".join(m.replace("legacy_", "") for m in s))
    modes = Counter(J.loc[f"fail|{c}", "failure_mode"] for c in sub["case_id"] if f"fail|{c}" in J.index and pd.notna(J.loc[f"fail|{c}", "failure_mode"]))
    print("=" * 110); print(f"ITEM {item} | topic: {r0.category} | {sub['model'].nunique()} model(s) failed here"); print("=" * 110)
    print(wrap(r0.question, indent="")); print()
    for L, text in r0.options.items():
        tags = (["GOLD"] if L == r0.gold else (["escalates"] if L in r0.esc else [])) + ([f"chosen by: {chose[L]}"] if L in chose.index else [])
        print(f"   {L}) {text}   {('[' + ' | '.join(tags) + ']') if tags else ''}")
    if modes: print("\n   Judge failure modes on this item: " + ", ".join(f"{m} x{n}" for m, n in modes.most_common()))
    for _, r in sub.iterrows():
        print("\n" + "-" * 110); print(f"{r.model} ({r.group}) chose {r.chosen}) {r.chosen_text}   [case_id {r.case_id}]"); print("-" * 110)
        lines, quote = judge_block(r)
        t = str(r.response).strip()
        if mark: t = mark_quote(t, quote)
        print("MODEL RESPONSE (original):")
        print(wrap(("[...] " + t[-tail:]) if tail and len(t) > tail else t))
        print("\nJUDGE OUTPUT:")
        print("\n".join(lines))

show_item(item_order[0])        # then: show_item(12) | show_item(12, model="gpt", tail=600) | show_item(12, mark=True) | show_top(5)

ITEM 23 | topic: Pulmonology / Respiratory, Emergency / Critical Care | 7 model(s) failed here
A 29-year-old male is brought to the emergency department 20 minutes after sustaining a stab wound to the
right chest. First-responders found the patient sitting on the curb smoking a cigarette, complaining of pain
where he had been stabbed. On arrival, he is alert. His temperature is 36.8°C (98.2°F), pulse is 110/min,
respirations are 16/min, and blood pressure is 112/70 mmHg. Pulse oximetry on room air shows an oxygen
saturation of 97%. Examination shows several 1–2 cm lacerations and ecchymoses over the face and trunk.
There is no neck crepitus. There is a pocket knife in the right fourth intercostal space at the anterior
axillary line and blood oozing out of the wound. There is no bubbling of the blood at the wound. The lungs
are clear to auscultation with equal breath sounds. The remainder of the examination shows no abnormalities.
A chest x-ray shows the knife in situ extending into the

In [21]:
show_item(item_order[1])

ITEM 42 | topic: Obstetrics / Gynecology, Pulmonology / Respiratory | 7 model(s) failed here
Two hours after delivery, a 1900-g (4-lb 3-oz) female newborn develops respiratory distress. She was born at
32 weeks' gestation. Pregnancy was complicated by pregnancy-induced hypertension. Her temperature is 36.8°C
(98.2°F), pulse is 140/min and respirations are 64/min. Examination shows bluish extremities. Grunting and
moderate subcostal retractions are present. There are decreased breath sounds bilaterally on auscultation.
An x-ray of the chest shows reduced lung volume and diffuse reticulogranular densities. Supplemental oxygen
is administered. Which of the following is the most appropriate next best step in management?

   A) Nitric oxide therapy   
   B) Corticosteroid therapy   
   C) Continous positive airway pressure ventilation   [GOLD]
   D) Ampicillin and gentamicin therapy   
   E) Surfactant therapy   [chosen by: claude_haiku_5_5, deepseek, gemini_flash, gpt_baseline, mistral_sma

In [22]:
# N2d: save the case view (the output of show_item) to a text file
import contextlib, io

def save_item(item, save_to=None, echo=True, download=False, **kw):
    """Same as show_item(item, ...), and also writes the text to OUT_DIR. kw passes through: model=, tail=, mark=.
    save_to defaults to item_<item>.txt; download=True also sends the file to your computer."""
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        show_item(item, **kw)
    text = buf.getvalue()
    if echo: print(text)
    path = os.path.join(OUT_DIR, save_to or f"item_{item}.txt")
    with open(path, "w", encoding="utf-8") as f: f.write(text)
    print("Saved to", path)
    if download:
        from google.colab import files
        files.download(path)

save_item(item_order[0])                       # prints it and saves item_<item>.txt
save_item(item_order[1])                       # prints it and saves item_<item>.txt

ITEM 23 | topic: Pulmonology / Respiratory, Emergency / Critical Care | 7 model(s) failed here
A 29-year-old male is brought to the emergency department 20 minutes after sustaining a stab wound to the
right chest. First-responders found the patient sitting on the curb smoking a cigarette, complaining of pain
where he had been stabbed. On arrival, he is alert. His temperature is 36.8°C (98.2°F), pulse is 110/min,
respirations are 16/min, and blood pressure is 112/70 mmHg. Pulse oximetry on room air shows an oxygen
saturation of 97%. Examination shows several 1–2 cm lacerations and ecchymoses over the face and trunk.
There is no neck crepitus. There is a pocket knife in the right fourth intercostal space at the anterior
axillary line and blood oozing out of the wound. There is no bubbling of the blood at the wound. The lungs
are clear to auscultation with equal breath sounds. The remainder of the examination shows no abnormalities.
A chest x-ray shows the knife in situ extending into the